[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_01_clustering.ipynb)

# S09 · 01 — Segmentar huéspedes con clustering

Sprint 9 · M3 (UD 3.9, clustering)

**Contexto TurisData.** TurisData Canarias tiene 1 200 huéspedes y hasta ahora les manda a todos la misma newsletter. Dirección pregunta: *¿hay grupos de clientes parecidos entre sí, y merece la pena tratarlos de forma distinta?* Aquí **no hay etiquetas**: nadie nos ha dicho a qué grupo pertenece cada huésped. Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego **nosotros** interpretamos qué significa cada grupo.

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 1 y 2 son los mismos que los del cuaderno «en clase» (allí, del 1 al 2).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
print(huespedes.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Las primeras filas de huespedes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas columnas tiene la tabla y cuáles son numéricas?
2. ¿Qué variable solo puede valer 0 ó 1?
3. ¿Hay alguna columna que diga a qué grupo pertenece el huésped?

In [ ]:
huespedes.head()

### 👀 A3 · EJEMPLO — Las escalas de las variables

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Entre qué valores va `gasto_medio_dia`?
2. ¿Qué variable varía más en unidades absolutas y cuál menos?
3. Si agrupamos por distancia sin tocar estas escalas, ¿qué variables mandarían?

In [ ]:
huespedes.describe().round(1).T

### 👀 A4 · EJEMPLO — Estandarizar una sola columna

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué media y qué desviación tiene la edad estandarizada?
2. Un huésped de 53 años (la media es 41,2 y la desviación 12,1), ¿cuánto vale su edad escalada?
3. ¿Qué consigue esto para el clustering?

In [ ]:
edad = huespedes["edad"]
edad_escalada = (edad - edad.mean()) / edad.std()
print("media:", round(edad_escalada.mean(), 3), "· desviación:", round(edad_escalada.std(), 3))

### ✏️ A5 · EJERCICIO 1 de 5 — La matriz X estandarizada

**Tu turno:** Crea la matriz `X` con **todas las variables menos `id_huesped`**, estandarizadas con `StandardScaler` (usa `fit_transform`). `X` debe ser una matriz de 1200 filas y 7 columnas. Guarda también los nombres de las variables en la lista `variables`. Ojo: más abajo (apartado 3) se añade a la tabla la columna `grupo`; si repites esta celda después, **no la incluyas** en `variables` (o reinicia el cuaderno).

**Debe salir:** ✔ Ejercicio 1 correcto (`X` tiene 1200 filas y 7 columnas, todas con media 0 y desviación 1)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Por qué se quita `id_huesped`?
2. ¿Qué tipo de objeto es `X` y qué ha cambiado respecto a `huespedes`?
3. ¿Qué media y desviación tiene cada columna de `X`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: quita la columna id_huesped con drop(columns=...) y pásale el resultado a StandardScaler().fit_transform
variables = ...
X = ...

assert X.shape == (1200, 7), "X debe tener 1200 filas y 7 columnas (sin id_huesped ni grupo)"
assert abs(X.mean()) < 1e-6 and abs(X[:, 0].std() - 1) < 1e-6, "Revisa que estés escalando con StandardScaler"
print("✔ Ejercicio 1 correcto")

### 👀 A6 · EJEMPLO — k-means con k = 3

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos huéspedes hay en cada uno de los tres grupos?
2. ¿Qué decide el usuario en k-means y qué decide el algoritmo?
3. ¿Por qué se pone `random_state=SEMILLA`?

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)
print("Tamaño de cada grupo:", np.bincount(km3.labels_))
print("Inercia (suma de distancias al centro):", round(km3.inertia_))

### ✏️ A7 · EJERCICIO 2 de 5 — ¿Cuántos grupos? La silueta de cada k

**Tu turno:** Calcula en el diccionario `siluetas` la silueta media (`silhouette_score(X, etiquetas)`) de k-means para cada `k` de 2 a 8 (clave = `k`, valor = silueta). Después guarda en `k_mejor` el `k` con silueta más alta. Usa `n_init=10` y `random_state=SEMILLA`.

**Debe salir:** `k con mejor silueta: 4` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué k tiene la mejor silueta y cuánto vale?
2. ¿Cómo de buena es esa estructura según la escala del texto?
3. ¿Por qué `max(siluetas)` no daría el mejor k?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: bucle for k in range(2, 9): entrena KMeans, saca .labels_ y calcula silhouette_score; luego max(siluetas, key=siluetas.get)
siluetas = {}
k_mejor = ...

assert set(siluetas) == set(range(2, 9)), "siluetas debe tener las claves 2, 3, ..., 8"
assert 3 <= k_mejor <= 5, "Revisa: k_mejor debería ser el k con la silueta más alta (está entre 3 y 5)"
print("k con mejor silueta:", k_mejor)
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Dibujar la silueta de cada k

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿En qué k está el máximo de la curva?
2. ¿Con 3, 4 o 5 grupos las siluetas son muy distintas?
3. ¿Qué pasa a partir de k = 6?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(siluetas), list(siluetas.values()), marker="o")
ax.axvline(k_mejor, color="tab:red", linestyle="--", label=f"mejor k = {k_mejor}")
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta media")
ax.set_title("¿Cuántos segmentos de huéspedes hay?")
ax.legend()
plt.show()

### 👀 A9 · EJEMPLO — El perfil medio de cada grupo

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Cuántos huéspedes tiene cada grupo?
2. ¿Qué grupo gasta más al día y cuánto?
3. ¿Qué grupo viaja casi siempre con niños?

In [ ]:
km = KMeans(n_clusters=k_mejor, n_init=10, random_state=SEMILLA).fit(X)
huespedes["grupo"] = km.labels_
perfil = huespedes.groupby("grupo")[variables].mean().round(1)
perfil["n_huespedes"] = huespedes["grupo"].value_counts().sort_index()
perfil

### 👀 A10 · EJEMPLO — Mapa de calor de los perfiles

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Qué grupo se define por viajar con niños?
2. ¿Qué dos variables destacan en el grupo 2?
3. ¿Qué distingue al grupo 0 (el de 194 huéspedes)?

In [ ]:
# Mapa de calor con los perfiles estandarizados (comparables entre variables)
perfil_z = pd.DataFrame(km.cluster_centers_, columns=variables)
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(perfil_z, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Perfil de cada grupo (en desviaciones típicas respecto a la media)")
ax.set_ylabel("Grupo")
plt.show()

### ✏️ A11 · EJERCICIO 3 de 5 — Familias y aventureros

**Tu turno:** Guarda en `grupo_familias` el número del grupo con **mayor proporción de huéspedes que viajan con niños** (media de `viaja_con_ninos` por grupo) y en `grupo_aventura` el número del grupo con **mayor `pct_gasto_actividades`** medio.

**Debe salir:** ✔ Ejercicio 3 correcto (`grupo_familias` = 1 y `grupo_aventura` = 3)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Qué valen `grupo_familias` y `grupo_aventura`?
2. ¿Qué diferencia hay entre `.max()` e `.idxmax()` aquí?
3. ¿Qué proporción de huéspedes con niños tiene cada uno de los otros tres grupos?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A11, **después de intentarlo**.

In [ ]:
# pista: groupby("grupo")[columna].mean() y luego .idxmax()
grupo_familias = ...
grupo_aventura = ...

assert huespedes[huespedes["grupo"] == grupo_familias]["viaja_con_ninos"].mean() > 0.6, "grupo_familias: busca el grupo donde casi todos viajan con niños"
assert huespedes[huespedes["grupo"] == grupo_aventura]["pct_gasto_actividades"].mean() > 50, "grupo_aventura: busca el grupo que más gasta en actividades"
print("✔ Ejercicio 3 correcto")

### 👀 A12 · EJEMPLO — El dendrograma de 150 huéspedes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Cuántas ramas de color tiene y qué significa?
2. ¿Qué indica que las últimas fusiones estén tan arriba (17 a 29)?
3. ¿Por qué se dibuja con una muestra de 150 y no con los 1 200?

In [ ]:
muestra = np.random.default_rng(SEMILLA).choice(len(X), size=150, replace=False)
Z = linkage(X[muestra], method="ward")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, no_labels=True, color_threshold=Z[-4, 2], ax=ax)
ax.set_title("Dendrograma (150 huéspedes): los saltos grandes indican grupos distintos")
ax.set_ylabel("Distancia de fusión")
plt.show()

### ✏️ A13 · EJERCICIO 4 de 5 — Clustering jerárquico con 4 grupos

**Tu turno:** Aplica `AgglomerativeClustering` con `n_clusters=4` y `linkage="ward"` sobre **todos** los huéspedes (`X`). Guarda las etiquetas en `etiquetas_jer` y su silueta en `silueta_jer`.

**Debe salir:** `Silueta jerárquico = 0.424 · k-means con k=4: 0.431` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Qué silueta tiene el jerárquico y cuál k-means con k = 4?
2. ¿Cuál de los dos agrupa «mejor» según la silueta?
3. ¿Qué ventaja tiene el jerárquico sobre k-means?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A13, **después de intentarlo**.

In [ ]:
# pista: AgglomerativeClustering se configura con n_clusters y linkage; la silueta se calcula sobre X y las etiquetas
etiquetas_jer = ...
silueta_jer = ...

assert len(etiquetas_jer) == 1200 and len(set(etiquetas_jer)) == 4, "Debe haber 4 grupos y una etiqueta por huésped"
assert silueta_jer > 0.3, "La silueta debería ser superior a 0,3; revisa que uses X (escalado)"
print(f"Silueta jerárquico = {silueta_jer:.3f} · k-means con k={k_mejor}: {siluetas[k_mejor]:.3f}")
print("✔ Ejercicio 4 correcto")

### 👀 A14 · EJEMPLO — Probar varios radios eps

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A14).

1. ¿Qué ocurre con el ruido cuando `eps` crece?
2. ¿Con qué `eps` sale el número de grupos más alto?
3. ¿Por qué se dice que DBSCAN es «sensible a eps»?

In [ ]:
for eps in (0.6, 0.8, 1.0, 1.2, 1.5):
    et = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    n_grupos = len(set(et)) - (1 if -1 in et else 0)
    print(f"eps={eps}: {n_grupos} grupos, {np.mean(et == -1):.0%} de ruido")

### ✏️ A15 · EJERCICIO 5 de 5 — DBSCAN con eps = 1,0

**Tu turno:** Entrena DBSCAN con `eps=1.0` y `min_samples=10`. Guarda las etiquetas en `etiquetas_db`, el número de grupos encontrados (sin contar el ruido) en `n_grupos_db` y cuántos huéspedes quedan como ruido en `n_ruido`.

**Debe salir:** `DBSCAN: 5 grupos y 149 huéspedes atípicos` y ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A15).

1. ¿Cuántos grupos y cuántos huéspedes atípicos salen?
2. ¿Por qué se resta 1 al contar los grupos?
3. Uno de los cinco grupos es muy pequeño (11 huéspedes), ¿qué significa?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A15, **después de intentarlo**.

In [ ]:
# pista: el ruido tiene etiqueta -1: cuéntalo con (etiquetas == -1).sum(); los grupos son las etiquetas distintas menos el -1
etiquetas_db = ...
n_grupos_db = ...
n_ruido = ...

assert n_grupos_db >= 3, "Con eps=1.0 debería encontrar varios grupos; revisa que uses X escalado"
assert 0 < n_ruido < 400, "n_ruido debe ser el número de huéspedes con etiqueta -1"
print(f"DBSCAN: {n_grupos_db} grupos y {n_ruido} huéspedes atípicos")
print("✔ Ejercicio 5 correcto")

### ✏️ A16 · MINI-RETO — Ponle nombre a tus segmentos

**Tu turno:** 1. Elige el método que te parezca mejor (k-means, jerárquico o DBSCAN) y justifícalo en una frase. 2. Para cada grupo, escribe **un nombre**, **una descripción de una línea** y **una acción comercial**. 3. Guarda los nombres en el diccionario `nombres_segmentos` (clave = número de grupo de k-means). *Ejemplo:* `{0: "Familias de verano"}`. Este trabajo alimenta el reto "Conocer al huésped".

**Debe salir:** ✔ Mini-reto correcto y el diccionario impreso, p. ej. `{0: 'Escapadas cortas', 1: 'Familias', 2: 'Descanso premium', 3: 'Aventureros'}` (tus nombres pueden ser otros)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A16).

1. ¿Qué nombre le darías al grupo 3 y por qué?
2. ¿Qué acción comercial propondrías para «Familias» (402 huéspedes)?
3. ¿Por qué los nombres deben salir del perfil y no de una intuición previa?

In [ ]:
# pista: mira el mapa de calor (perfil_z): edad, gasto, niños y % de gasto en actividades dan mucha pista
nombres_segmentos = {}   # rellena: {0: "...", 1: "...", ...}

assert len(nombres_segmentos) == k_mejor, "Pon un nombre a cada uno de los grupos de k-means"
print(nombres_segmentos)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn` y `scipy`, el escalado, los tres algoritmos de agrupación (`KMeans`, `AgglomerativeClustering`, `DBSCAN`), la silueta y el dendrograma.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Fija la semilla `SEMILLA = 42` (para que todos obtengan los mismos grupos) y lee `huespedes.csv` en `huespedes`: 1 200 huéspedes con un identificador y 7 variables de comportamiento.
- **Qué debe salir:**
    - `(1200, 8)` (1 200 filas y 8 columnas) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Escalar: que todas las variables «pesen» lo mismo

#### A2 · Ejemplo — Las primeras filas de huespedes

**Contexto.** No hay etiquetas: nadie nos dice a qué grupo pertenece cada huésped. Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego **nosotros** interpretamos qué significa cada grupo. Cada fila es un huésped con 7 variables de comportamiento (más su identificador).

- **Qué hace:**
    - `huespedes.head()` muestra las 5 primeras filas de la tabla.
- **Qué debe salir:**
    - Una tabla con 5 filas y 8 columnas: `id_huesped`, `edad`, `gasto_medio_dia`, `noches_medias`, `reservas_anio`, `antelacion_media`, `pct_gasto_actividades` y `viaja_con_ninos`. El primer huésped tiene 26 años, gasta 92,25 € al día y se aloja 3,9 noches.
- **Cómo leerlo:**
    - Cada fila es un huésped y cada columna una variable de su comportamiento. `viaja_con_ninos` vale 0 ó 1; `pct_gasto_actividades`, de 0 a 100.
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene la tabla y cuáles son numéricas?* → **8** columnas; las 8 son numéricas (una es el identificador, que no describe el comportamiento).
2. *¿Qué variable solo puede valer 0 ó 1?* → `viaja_con_ninos`: 1 si el huésped viaja con niños y 0 si no.
3. *¿Hay alguna columna que diga a qué grupo pertenece el huésped?* → **No**: no hay etiquetas. Por eso es aprendizaje **no supervisado**: los grupos los tendrá que encontrar el algoritmo.

</details>

#### A3 · Ejemplo — Las escalas de las variables

Fíjate en las **escalas**: la edad va de 18 a 75, el gasto diario de 35 a 340 y `viaja_con_ninos` solo vale 0 ó 1.

- **Qué hace:**
    - `huespedes.describe()` calcula, para cada columna, el recuento, la media, la desviación típica, el mínimo, los cuartiles y el máximo; `.round(1)` los deja con un decimal y `.T` traspone la tabla (una fila por variable).
- **Qué debe salir:**
    - Una tabla con una fila por variable. La media y la desviación típica de las más relevantes: `edad` 41,2 y 12,1 · `gasto_medio_dia` 124,2 y 60,9 · `antelacion_media` 78,7 y 36,5 · `viaja_con_ninos` 0,3 y 0,5.
- **Cómo leerlo:**
    - Compara la columna `std` (cuánto varía cada variable): el gasto y la antelación varían decenas de unidades; los niños, medio punto.
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué valores va `gasto_medio_dia`?* → De **34,6** a **338,7** euros al día (con una media de 124,2).
2. *¿Qué variable varía más en unidades absolutas y cuál menos?* → `gasto_medio_dia` y `antelacion_media` son las que más varían (`std` 60,9 y 36,5); `viaja_con_ninos` la que menos (0,5).
3. *Si agrupamos por distancia sin tocar estas escalas, ¿qué variables mandarían?* → Las de **números grandes**, es decir el gasto y la antelación; `viaja_con_ninos` apenas contaría. Por eso hay que escalar.

</details>

#### A4 · Ejemplo — Estandarizar una sola columna

**Idea.** Clustering agrupa por **distancia**. Si una variable tiene números grandes (gasto en euros) y otra pequeños (0/1 de niños), la grande manda y la pequeña no cuenta. Solución: **estandarizar** (restar la media y dividir por la desviación típica) para que todas tengan media 0 y desviación 1.

- **Qué hace:**
    - `edad = huespedes["edad"]` toma la columna de edades.
    - `(edad - edad.mean()) / edad.std()` resta la media y divide entre la desviación típica: es la estandarización a mano.
    - El `print` comprueba que la media vale 0 y la desviación 1, redondeadas a 3 decimales.
- **Qué debe salir:**
    - `media: 0.0 · desviación: 1.0`.
- **Cómo leerlo:**
    - Después de estandarizar, 0 significa «en la media» y 1 «una desviación típica por encima»: la edad ya no se mide en años sino en «desviaciones típicas».
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué media y qué desviación tiene la edad estandarizada?* → Media **0** y desviación **1**.
2. *Un huésped de 53 años (la media es 41,2 y la desviación 12,1), ¿cuánto vale su edad escalada?* → (53 − 41,2) / 12,1 ≈ **+1,0**: está una desviación típica por encima de la media.
3. *¿Qué consigue esto para el clustering?* → Que **todas las variables pesen lo mismo** al calcular distancias, sea cual sea su unidad (años, euros, 0/1).

</details>

#### A5 · Ejercicio 1 — La matriz X estandarizada

- **Qué hace:**
    - `variables` es la lista de nombres de columna sin `id_huesped` (el identificador no describe el comportamiento) ni `grupo` (esa columna se crea más adelante; así puedes repetir esta celda).
    - `StandardScaler().fit_transform(huespedes[variables])` estandariza cada columna (media 0, desviación 1) y devuelve una **matriz** de números (`X`), ya sin nombres de columna.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `X debe tener 1200 filas y 7 columnas` → has dejado `id_huesped` (8 columnas) o has guardado en `X` otra cosa que la matriz estandarizada.
    - `Revisa que estés escalando con StandardScaler` → `X` no tiene media 0 y desviación 1: has pasado los datos sin escalar.
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se quita `id_huesped`?* → Porque es solo un número de identificación: **no describe el comportamiento** y, si lo dejáramos, el algoritmo agruparía por un número que no significa nada.
2. *¿Qué tipo de objeto es `X` y qué ha cambiado respecto a `huespedes`?* → Una **matriz** (array de NumPy) de 1200 × 7, **sin nombres de columna** y con todas las variables estandarizadas. Los nombres se guardan aparte en `variables`.
3. *¿Qué media y desviación tiene cada columna de `X`?* → **Media 0 y desviación 1** (así lo comprueba el `assert`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
variables = [c for c in huespedes.columns if c not in ("id_huesped", "grupo")]   # "grupo" aparece más abajo; así puedes repetir esta celda
X = StandardScaler().fit_transform(huespedes[variables])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. k-means: agrupar alrededor de centros

#### A6 · Ejemplo — k-means con k = 3

**Idea.** Le decimos cuántos grupos queremos (`k`). El algoritmo coloca `k` centros, asigna cada huésped al centro más cercano, recalcula los centros y repite hasta que no cambian. Es rápido y fácil de explicar. Su límite: **tú decides `k`** y prefiere grupos «redondos» y de tamaño parecido.

- **Qué hace:**
    - `KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)` agrupa los huéspedes en 3 grupos (`n_init=10` repite el proceso 10 veces desde centros distintos y se queda con el mejor).
    - `np.bincount(km3.labels_)` cuenta cuántos huéspedes hay en cada grupo.
    - `km3.inertia_` es la **inercia**: la suma de las distancias al cuadrado de cada huésped a su centro (cuanto menor, grupos más compactos).
- **Qué debe salir:**
    - `Tamaño de cada grupo: [317 480 403]` e `Inercia (suma de distancias al centro): 3487`.
- **Cómo leerlo:**
    - Los tres números del primer `print` suman 1 200 (317 + 480 + 403): cada huésped está en un solo grupo. La inercia solo sirve para comparar modelos con el mismo `k`: siempre baja al aumentar `k`.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos huéspedes hay en cada uno de los tres grupos?* → **317, 480 y 403** (suman 1 200).
2. *¿Qué decide el usuario en k-means y qué decide el algoritmo?* → El usuario decide **cuántos grupos (`k`)**; el algoritmo decide **quién va a cada grupo** moviendo los centros hasta que se estabilizan.
3. *¿Por qué se pone `random_state=SEMILLA`?* → Para que el resultado sea **reproducible**: k-means empieza con centros al azar, y con la misma semilla todos obtienen los mismos grupos.

</details>

#### A7 · Ejercicio 2 — ¿Cuántos grupos? La silueta de cada k

**El coeficiente de silueta.** Para cada huésped compara *lo cerca que está de su grupo* con *lo cerca que está del grupo vecino*. Va de −1 a 1: **cuanto más alto, grupos más compactos y separados**. Por encima de 0,5 se considera una buena estructura; entre 0,25 y 0,5, aceptable.

- **Qué hace:**
    - `for k in range(2, 9)` prueba de 2 a 8 grupos. Para cada `k`: `KMeans(...).fit_predict(X)` agrupa y devuelve la etiqueta de cada huésped, y `silhouette_score(X, etiquetas)` calcula la silueta media.
    - `siluetas[k] = ...` guarda el resultado en el diccionario (clave = `k`, valor = silueta).
    - `max(siluetas, key=siluetas.get)` devuelve la **clave** con mayor valor, es decir el mejor `k` (`k_mejor`).
- **Qué debe salir:**
    - `k con mejor silueta: 4` y `✔ Ejercicio 2 correcto`. Las siluetas valen: 2 → 0,367 · 3 → 0,415 · **4 → 0,431** · 5 → 0,414 · 6 → 0,326 · 7 → 0,317 · 8 → 0,257.
- **Si falla:**
    - `siluetas debe tener las claves 2, 3, ..., 8` → el diccionario está vacío o el bucle no recorre `range(2, 9)`.
    - `Revisa: k_mejor debería ser el k con la silueta más alta` → `k_mejor` no es una clave del diccionario con mayor silueta (¿has cogido `max(siluetas)`, que da la mayor **clave**, 8, en lugar de usar `key=siluetas.get`?).
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué k tiene la mejor silueta y cuánto vale?* → **k = 4**, con una silueta de **0,431**.
2. *¿Cómo de buena es esa estructura según la escala del texto?* → **Aceptable** (entre 0,25 y 0,5): hay grupos, pero no están muy separados; no llega a 0,5.
3. *¿Por qué `max(siluetas)` no daría el mejor k?* → Porque `max` sobre un diccionario compara las **claves** (2 a 8) y devolvería 8. Hay que decirle que compare los **valores**: `key=siluetas.get`.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
siluetas = {}
for k in range(2, 9):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit_predict(X)
    siluetas[k] = silhouette_score(X, etiquetas)
k_mejor = max(siluetas, key=siluetas.get)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Dibujar la silueta de cada k

- **Qué hace:**
    - `ax.plot(list(siluetas), list(siluetas.values()), marker="o")` dibuja la silueta (eje y) frente a `k` (eje x), con un punto por cada `k`.
    - `ax.axvline(k_mejor, ...)` traza una línea roja discontinua en el mejor `k`.
    - El resto pone títulos, etiquetas de ejes y leyenda (`plt.show()` muestra la figura).
- **Qué debe salir:**
    - Una curva con 7 puntos: sube de 0,37 (k = 2) a 0,43 (k = 4), baja a 0,41 (k = 5) y cae hasta 0,26 (k = 8). La línea roja marca k = 4.
- **Cómo leerlo:**
    - Se busca el **máximo** de la curva. Aquí es poco marcado: k = 3, 4 y 5 dan siluetas parecidas (0,415, 0,431 y 0,414); a partir de 6 hay una caída clara.
- **Si falla:**
    - `NameError: name 'siluetas' is not defined` → falta ejecutar el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué k está el máximo de la curva?* → En **k = 4** (silueta 0,431).
2. *¿Con 3, 4 o 5 grupos las siluetas son muy distintas?* → **No**: 0,415, 0,431 y 0,414. La diferencia es pequeña, así que elegir entre 3, 4 o 5 es una decisión que también puede apoyarse en el negocio.
3. *¿Qué pasa a partir de k = 6?* → La silueta **cae** claramente (0,326, 0,317, 0,257): con tantos grupos se parten grupos que en realidad eran uno solo.

</details>

## 3. Interpretar: ¿quién es cada grupo?

#### A9 · Ejemplo — El perfil medio de cada grupo

**Idea.** Un grupo sin nombre no sirve al negocio. Calculamos la **media de cada variable por grupo** y la leemos como si describiéramos a un cliente típico.

- **Qué hace:**
    - `KMeans(n_clusters=k_mejor, ...)` agrupa en 4 grupos y `huespedes["grupo"] = km.labels_` añade a la tabla el grupo de cada huésped.
    - `huespedes.groupby("grupo")[variables].mean().round(1)` calcula la media de cada variable en cada grupo (`perfil`).
    - `perfil["n_huespedes"]` añade cuántos huéspedes tiene cada grupo.
- **Qué debe salir:**
    - Una tabla de 4 filas (grupos 0 a 3). Por ejemplo, el grupo 0 tiene 194 huéspedes con 2,6 noches y 4,0 reservas al año de media; el grupo 2, 317 huéspedes de 57 años que gastan 211 € al día.
- **Cómo leerlo:**
    - Lee cada fila como un cliente típico: el grupo 1 (402 huéspedes) viaja con niños (0,9) y reserva con mucha antelación (110 días); el grupo 3 (287), el más joven (29 años), gasta el 75 % en actividades.
- **Si falla:**
    - `NameError: name 'k_mejor' is not defined` → falta el ejercicio 2. Si repites esta celda, se vuelve a añadir `grupo` sin problema.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos huéspedes tiene cada grupo?* → Grupo 0: **194** · grupo 1: **402** · grupo 2: **317** · grupo 3: **287**.
2. *¿Qué grupo gasta más al día y cuánto?* → El **grupo 2**, con **211,4 €** de media (y una edad media de 57 años).
3. *¿Qué grupo viaja casi siempre con niños?* → El **grupo 1**: `viaja_con_ninos` vale 0,9, es decir el 90 % de sus huéspedes.

</details>

#### A10 · Ejemplo — Mapa de calor de los perfiles

Las medias de la tabla tienen unidades distintas (años, euros, noches…), así que no se pueden comparar de una variable a otra. Los centros de k-means, que están en la escala estandarizada, sí.

- **Qué hace:**
    - `pd.DataFrame(km.cluster_centers_, columns=variables)` convierte los centros de los 4 grupos (en desviaciones típicas) en una tabla (`perfil_z`).
    - `sns.heatmap(..., annot=True, fmt=".1f", cmap="RdBu_r", center=0)` la dibuja como mapa de calor: rojo = por encima de la media, azul = por debajo, con el número escrito en cada casilla.
- **Qué debe salir:**
    - Un mapa de calor de 4 filas × 7 columnas. Casillas destacadas: grupo 0 → `noches_medias` −1,7, `antelacion_media` −1,5 y `reservas_anio` +1,2; grupo 1 → `viaja_con_ninos` +1,2; grupo 2 → `edad` +1,3 y `gasto_medio_dia` +1,4; grupo 3 → `pct_gasto_actividades` +1,5.
- **Cómo leerlo:**
    - Cada casilla dice cuántas desviaciones típicas está el grupo por encima (rojo) o por debajo (azul) de la media de todos los huéspedes. Las casillas más intensas son las que definen al grupo.
- **Si falla:**
    - `NameError: name 'km' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué grupo se define por viajar con niños?* → El **grupo 1** (`viaja_con_ninos` ≈ +1,2).
2. *¿Qué dos variables destacan en el grupo 2?* → La **edad** (+1,3) y el **gasto medio diario** (+1,4): clientes mayores que gastan mucho.
3. *¿Qué distingue al grupo 0 (el de 194 huéspedes)?* → **Estancias muy cortas** (`noches_medias` −1,7), **reservas de última hora** (`antelacion_media` −1,5) y **muchas reservas al año** (+1,2).

</details>

#### A11 · Ejercicio 3 — Familias y aventureros

- **Qué hace:**
    - `huespedes.groupby("grupo")["viaja_con_ninos"].mean()` da la proporción de huéspedes con niños en cada grupo; `.idxmax()` devuelve **el número de grupo** (el índice) con el valor más alto → `grupo_familias`.
    - Lo mismo con `pct_gasto_actividades` → `grupo_aventura`.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. Los valores son `grupo_familias = 1` (91 % viaja con niños) y `grupo_aventura = 3` (74,6 % de gasto en actividades).
- **Si falla:**
    - `grupo_familias: busca el grupo donde casi todos viajan con niños` → has guardado el valor máximo (`.max()`) y no el número de grupo (`.idxmax()`), o has agrupado por otra columna.
    - `grupo_aventura: busca el grupo que más gasta en actividades` → igual con `pct_gasto_actividades`.
    - `KeyError: 'grupo'` → falta el ejemplo del perfil, que crea la columna `grupo`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `grupo_familias` y `grupo_aventura`?* → **1** y **3**.
2. *¿Qué diferencia hay entre `.max()` e `.idxmax()` aquí?* → `.max()` da **el valor** más alto (por ejemplo 0,91) e `.idxmax()` da **el grupo** donde está (1). Aquí queremos el grupo.
3. *¿Qué proporción de huéspedes con niños tiene cada uno de los otros tres grupos?* → Muy pocos: **0,06, 0,02 y 0,05**. El grupo 1 es el único «familias».

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
grupo_familias = huespedes.groupby("grupo")["viaja_con_ninos"].mean().idxmax()
grupo_aventura = huespedes.groupby("grupo")["pct_gasto_actividades"].mean().idxmax()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Clustering jerárquico: un árbol de parecidos

#### A12 · Ejemplo — El dendrograma de 150 huéspedes

**Idea.** Empieza con cada huésped como su propio grupo y va **fusionando los dos más parecidos** paso a paso. El resultado es un árbol (**dendrograma**): cuanto más alto es el salto entre fusiones, más distintos eran los grupos que se unían. Se «corta» el árbol a la altura que queramos. Con `linkage="ward"` se busca que los grupos sean compactos.

- **Qué hace:**
    - `np.random.default_rng(SEMILLA).choice(len(X), size=150, replace=False)` elige 150 huéspedes al azar sin repetir (con los 1 200 sería ilegible).
    - `linkage(X[muestra], method="ward")` construye el árbol de fusiones (`Z`).
    - `dendrogram(Z, no_labels=True, color_threshold=Z[-4, 2], ...)` lo dibuja; `color_threshold` corta a la altura de la 4.ª fusión desde el final, que deja **4 ramas de colores**.
- **Qué debe salir:**
    - Un dendrograma de 150 hojas con 4 ramas de color. Las tres últimas fusiones están muy arriba (alturas ≈ 17,5, 20,4 y 29,3) y la anterior en ≈ 8,7.
- **Cómo leerlo:**
    - Fíjate en el **eje vertical**: los saltos grandes al final indican que lo que se fusiona ahí es muy distinto. Cortar por debajo de ellos da 4 grupos.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas ramas de color tiene y qué significa?* → **4**: son los 4 grupos que quedan al cortar el árbol en la altura marcada.
2. *¿Qué indica que las últimas fusiones estén tan arriba (17 a 29)?* → Que los grupos que se unen ahí son **muy distintos entre sí**: unirlos obligaría a mezclar huéspedes poco parecidos.
3. *¿Por qué se dibuja con una muestra de 150 y no con los 1 200?* → Porque con 1 200 hojas el dendrograma sería **ilegible**; la muestra basta para ver la estructura.

</details>

#### A13 · Ejercicio 4 — Clustering jerárquico con 4 grupos

- **Qué hace:**
    - `AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X)` agrupa a **todos** los huéspedes en 4 grupos y devuelve la etiqueta de cada uno (`etiquetas_jer`).
    - `silhouette_score(X, etiquetas_jer)` calcula su silueta media (`silueta_jer`).
    - El `print` la compara con la de k-means con k = 4.
- **Qué debe salir:**
    - `Silueta jerárquico = 0.424 · k-means con k=4: 0.431` y `✔ Ejercicio 4 correcto`. Los 4 grupos tienen 414, 192, 307 y 287 huéspedes.
- **Si falla:**
    - `Debe haber 4 grupos y una etiqueta por huésped` → no has puesto `n_clusters=4` o has aplicado el método a otra cosa que `X`.
    - `La silueta debería ser superior a 0,3; revisa que uses X (escalado)` → has calculado la silueta con datos sin escalar o con otras etiquetas.
    - `NameError: name 'k_mejor' is not defined` → falta el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué silueta tiene el jerárquico y cuál k-means con k = 4?* → **0,424** el jerárquico y **0,431** k-means.
2. *¿Cuál de los dos agrupa «mejor» según la silueta?* → K-means, por muy poco (0,431 frente a 0,424): la diferencia es **tan pequeña** que en la práctica son equivalentes.
3. *¿Qué ventaja tiene el jerárquico sobre k-means?* → Que da el **dendrograma**: se puede ver cómo se van uniendo los grupos y decidir el corte mirándolo. A cambio es **más lento** con muchos datos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
etiquetas_jer = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X)
silueta_jer = silhouette_score(X, etiquetas_jer)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. DBSCAN: grupos por densidad (y ruido)

#### A14 · Ejemplo — Probar varios radios eps

**Idea.** No hay que decir cuántos grupos queremos. Define *«un grupo es una zona densa de huéspedes»* con dos parámetros: `eps` (radio de vecindad) y `min_samples` (cuántos vecinos hacen falta para considerarla densa). Lo que queda en zonas poco densas se etiqueta **−1 = ruido** (huéspedes atípicos). Es útil para detectar **casos raros**, pero es sensible a `eps`.

- **Qué hace:**
    - Para cada `eps` entre 0,6 y 1,5: `DBSCAN(eps=eps, min_samples=10).fit_predict(X)` devuelve la etiqueta de cada huésped (−1 = ruido).
    - `n_grupos` cuenta las etiquetas distintas sin el −1; `np.mean(et == -1)` es la proporción de ruido.
- **Qué debe salir:**
    - `eps=0.6: 4 grupos, 94% de ruido` · `eps=0.8: 4 grupos, 38% de ruido` · `eps=1.0: 5 grupos, 12% de ruido` · `eps=1.2: 3 grupos, 4% de ruido` · `eps=1.5: 2 grupos, 2% de ruido`.
- **Cómo leerlo:**
    - Con un radio pequeño casi todo es ruido (94 %); con uno grande, todo se junta en 2 grupos. Hay que buscar un `eps` intermedio, como 1,0 (5 grupos y 12 % de ruido).
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué ocurre con el ruido cuando `eps` crece?* → **Baja** (de 94 % a 2 %): con un radio mayor, más huéspedes tienen vecinos suficientes y entran en algún grupo.
2. *¿Con qué `eps` sale el número de grupos más alto?* → Con **eps = 1.0**: 5 grupos (y 12 % de ruido).
3. *¿Por qué se dice que DBSCAN es «sensible a eps»?* → Porque pequeños cambios del radio cambian **mucho** el resultado: de 94 % de ruido (0,6) a 12 % (1,0) o a 2 grupos (1,5).

</details>

#### A15 · Ejercicio 5 — DBSCAN con eps = 1,0

- **Qué hace:**
    - `DBSCAN(eps=1.0, min_samples=10).fit_predict(X)` agrupa por densidad y da la etiqueta de cada huésped (`etiquetas_db`; −1 = ruido).
    - `len(set(etiquetas_db)) - (1 if -1 in etiquetas_db else 0)` cuenta las etiquetas distintas **sin contar el ruido** (`n_grupos_db`).
    - `(etiquetas_db == -1).sum()` cuenta los huéspedes de etiqueta −1 (`n_ruido`); `int(...)` lo pasa a número normal.
- **Qué debe salir:**
    - `DBSCAN: 5 grupos y 149 huéspedes atípicos` y `✔ Ejercicio 5 correcto`. Los 5 grupos tienen 352, 260, 153, 275 y 11 huéspedes.
- **Si falla:**
    - `Con eps=1.0 debería encontrar varios grupos; revisa que uses X escalado` → has pasado otros datos o no has puesto `eps=1.0` y `min_samples=10`.
    - `n_ruido debe ser el número de huéspedes con etiqueta -1` → `n_ruido` vale 0 o es otra cosa (¿has contado las etiquetas distintas, en lugar de las que valen −1?).
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos grupos y cuántos huéspedes atípicos salen?* → **5 grupos** y **149** huéspedes atípicos (12,4 %).
2. *¿Por qué se resta 1 al contar los grupos?* → Porque la etiqueta **−1 no es un grupo**: es el «cajón» del ruido. Si aparece, hay que descontarla.
3. *Uno de los cinco grupos es muy pequeño (11 huéspedes), ¿qué significa?* → Que DBSCAN ha encontrado una **zona densa pequeña**: puede ser un nicho real o simplemente un trozo de otro grupo; hay que mirarlo antes de actuar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
etiquetas_db = DBSCAN(eps=1.0, min_samples=10).fit_predict(X)
n_grupos_db = len(set(etiquetas_db)) - (1 if -1 in etiquetas_db else 0)
n_ruido = int((etiquetas_db == -1).sum())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Mini-reto: ponle nombre a tus segmentos

#### A16 · Mini-reto — Ponle nombre a tus segmentos

- **Qué hace:**
    - Recorre los `k_mejor` = 4 grupos; para cada uno toma su fila del mapa de calor (`perfil_z.iloc[g]`).
    - Reglas de la solución de referencia: niños > 0,8 → «Familias»; % de gasto en actividades > 0,8 → «Aventureros»; edad > 0,5 → «Descanso premium»; el resto → «Escapadas cortas».
    - Los nombres no tienen que coincidir con los de referencia: lo que se comprueba es que **hay un nombre por grupo** (4).
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y el diccionario `{0: 'Escapadas cortas', 1: 'Familias', 2: 'Descanso premium', 3: 'Aventureros'}` (en la solución de referencia).
- **Cómo leerlo:**
    - Tu justificación del método (una frase) y tus descripciones y acciones comerciales por grupo van en una celda de texto: no se autocorrigen. Por ejemplo, «Escapadas cortas» (194 huéspedes, 2,6 noches, reservan con poca antelación) → oferta de última hora; «Familias» (402) → paquetes con actividades para niños.
- **Si falla:**
    - `Pon un nombre a cada uno de los grupos de k-means` → `nombres_segmentos` no tiene tantas claves como `k_mejor` (4).
    - `NameError: name 'k_mejor' is not defined` → faltan los ejercicios anteriores (o el perfil, que crea los grupos).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué nombre le darías al grupo 3 y por qué?* → «**Aventureros**»: el más joven (29 años), gasta el **75 %** en actividades y poco al día (−0,9 desviaciones típicas).
2. *¿Qué acción comercial propondrías para «Familias» (402 huéspedes)?* → Por ejemplo, **paquetes con actividades para niños** y reserva anticipada (reservan con 110 días de antelación).
3. *¿Por qué los nombres deben salir del perfil y no de una intuición previa?* → Porque los grupos los han creado los **datos**: un nombre solo es válido si describe lo que de verdad hay en el grupo (tamaño, edad, gasto…).

</details>

## Preguntas de reflexión
1. ¿Qué pasaría con los resultados si no escalaras las variables? Pruébalo y compara los grupos y la silueta:
   una silueta más alta no garantiza que todas las columnas estén contando. *(pista: ¿qué columnas mandan?)*
2. ¿Por qué la silueta más alta no siempre da el mejor número de segmentos *para el negocio*?
3. ¿Qué método te parece más fácil de explicar a dirección? ¿Y el más útil para detectar clientes raros?
4. Un segmento "familias" sale de datos de comportamiento, no de una etiqueta oficial: ¿qué riesgos hay
   si el hotel usa el segmento para decidir a quién ofrece descuentos? *(pista: sesgos y discriminación)*.

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Mandarían las variables de **números grandes**: el gasto (desviación típica de 61) y la antelación (37) frente a la edad (12) y `viaja_con_ninos` (0,5), que apenas contaría. Al probarlo sin escalar la silueta con k = 4 sube a 0,51 (frente a 0,43), pero no significa que sea mejor: el algoritmo solo ve unas pocas columnas. La silueta compara grupos, no comprueba que **todas las columnas estén contando**.
2. Porque la silueta mide lo compactos que son los grupos, no lo **útiles** que son. Aquí k = 3, 4 y 5 dan siluetas casi iguales (0,415, 0,431, 0,414): elegir entre ellos depende de cuántos segmentos puede gestionar el hotel y de si se pueden describir con una acción distinta cada uno.
3. El **jerárquico** es el más fácil de explicar a dirección: el dendrograma muestra cómo se van uniendo los grupos. **DBSCAN** es el más útil para detectar clientes raros, porque marca como ruido (−1) a los atípicos (aquí, 149 huéspedes con `eps=1.0`).
4. Que el segmento es una **hipótesis estadística**, no una etiqueta oficial: puede dejar fuera o tratar peor a personas concretas (por ejemplo, ofrecer descuentos solo a «familias» deja sin ellos a quien viaja con niños pero tiene otro patrón). Además, los datos de comportamiento pueden reflejar **sesgos** (edad, composición familiar) y llevar a decisiones discriminatorias. Hay que revisar quién queda dentro y quién fuera y no usarlo como única razón.

</details>

## Qué has aprendido
- Escalar antes de medir distancias.
- **k-means** (rápido, exige `k`), **jerárquico** (dendrograma, más lento) y **DBSCAN** (densidad, detecta ruido).
- La **silueta** ayuda a elegir `k`, pero la interpretación la pone el negocio.
- Un segmento solo es útil si tiene nombre, tamaño y una acción asociada.